In [1]:
# Importation des librairies nécessaires
from pathlib import Path
import html
import re
import numpy as np
import pandas as pd
from ftfy import fix_text
from unidecode import unidecode
from lingua import Language, LanguageDetectorBuilder
import nltk
from nltk.tokenize import word_tokenize
import simplemma
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn import model_selection
from sklearn.model_selection import train_test_split
from sklearn.svm import LinearSVC
from sklearn.metrics import f1_score



In [2]:
# Chargement des fichiers CSV

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 300)

X_train_Rakuten=pd.read_csv('/Users/edernpamart/Desktop/FORMATION DATA SCIENTIST - LIORA /PROJET/data/X_train_update.csv',index_col=0)
print(X_train_Rakuten.shape)
print(X_train_Rakuten.columns.tolist())
print(X_train_Rakuten.head(30))
X_test_Rakuten=pd.read_csv('/Users/edernpamart/Desktop/FORMATION DATA SCIENTIST - LIORA /PROJET/data/X_test_update.csv',index_col=0)
print(X_test_Rakuten.shape)
print(X_test_Rakuten.columns.tolist())
print(X_test_Rakuten.head(30))
Y_train_CVw08PX=pd.read_csv('/Users/edernpamart/Desktop/FORMATION DATA SCIENTIST - LIORA /PROJET/data/Y_train_CVw08PX.csv',index_col=0)
print(Y_train_CVw08PX.shape)
print(Y_train_CVw08PX.columns.tolist())
print(Y_train_CVw08PX.head(30))

nb_image_train=len(list(Path("data/images/image_train").glob("*.jpg")))
print("Le nombre d'image dans le set de train est : ", nb_image_train)
nb_image_test=len(list(Path("data/images/image_test").glob("*.jpg")))
print("Le nombre d'image dans le set de test est : ", nb_image_test)


(84916, 4)
['designation', 'description', 'productid', 'imageid']
                                          designation                                        description   productid     imageid
0   Olivia: Personalisiertes Notizbuch / 150 Seite...                                                NaN  3804725264  1263597046
1   Journal Des Arts (Le) N° 133 Du 28/09/2001 - L...                                                NaN   436067568  1008141237
2   Grand Stylet Ergonomique Bleu Gamepad Nintendo...  PILOT STYLE Touch Pen de marque Speedlink est ...   201115110   938777978
3   Peluche Donald - Europe - Disneyland 2000 (Mar...                                                NaN    50418756   457047496
4                                La Guerre Des Tuques  Luc a des id&eacute;es de grandeur. Il veut or...   278535884  1077757786
5   Afrique Contemporaine N° 212 Hiver 2004 - Doss...                                                NaN     5862738   393356830
6          Christof E: Bildungs

In [3]:
# Première étape du préprocessing: le nettoyage des balise HTML dans les colonnes description et designation des dataframes X_train_Rakuten et X_test_Rakuten

def decoder_html(text): # Création d'une fonction pour décoder les entités HTML et corriger les caractères spéciaux mal encodés
    if isinstance(text, str):
        return fix_text(html.unescape(text)) # La fonction fix_text corrige les caractères spéciaux mal encodés & html.unescape convertit les entités HTML en caractères normaux
    else:
        return text

for dataframe in [X_train_Rakuten, X_test_Rakuten]:
    for colonne in ["description", "designation"]:
        dataframe[colonne] = dataframe[colonne].apply(decoder_html)
        dataframe[colonne] = dataframe[colonne].str.replace(r'<[^>]*>', ' ', regex=True) #remplace les balises HTML par un espace
        dataframe[colonne] = dataframe[colonne].str.replace(r"\s+", " ", regex=True).str.strip() #remplace les espaces multiples par un seul espace et supprime les espaces en début et fin de chaîne

print(X_train_Rakuten.head(40)) # Première vérification du nettoyage des colonnes description et designation du dataframe X_train_Rakuten



                                          designation                                        description   productid     imageid
0   Olivia: Personalisiertes Notizbuch / 150 Seite...                                                NaN  3804725264  1263597046
1   Journal Des Arts (Le) N° 133 Du 28/09/2001 - L...                                                NaN   436067568  1008141237
2   Grand Stylet Ergonomique Bleu Gamepad Nintendo...  PILOT STYLE Touch Pen de marque Speedlink est ...   201115110   938777978
3   Peluche Donald - Europe - Disneyland 2000 (Mar...                                                NaN    50418756   457047496
4                                La Guerre Des Tuques  Luc a des idées de grandeur. Il veut organiser...   278535884  1077757786
5   Afrique Contemporaine N° 212 Hiver 2004 - Doss...                                                NaN     5862738   393356830
6          Christof E: Bildungsprozessen Auf Der Spur                                            

In [4]:
# L'affichage des 30 premières lignes des colonnes description et designation du dataframe X_test_Rakuten pour vérifier le nettoyage des balises HTML 
# nous permet de constater que les texte est plus propre qu'avant, nous allons maintenant créer une fonction pour vérifier l'existance dans tous les dataframes 
# d'autres caractères spéciaux mal encodés et les corriger si nécessaire.

anomaliesàsupprimer = {"balises_html": r'<[^>]*>', "entites_html": r'&[a-zA-Z]+;', "caracteres_speciaux": r"[^a-zA-Z0-9\sàâäéèêëîïôöùûüçÀÂÄÉÈÊËÎÏÔÖÙÛÜÇ.,'\-:;!?()/%€&+°\"]"
, "espaces_inutiles": r'\s{2,}'} #Liste des anomalies à supprimer avec les expressions correspondantes

for nom_dataframe, dataframe in [("X_train", X_train_Rakuten), ("X_test", X_test_Rakuten)]: #Boucle sur les dataframes X_train_Rakuten et X_test_Rakuten pour vérifier la présence d'anomalies dans les colonnes description et designation
    for colonne in ["description", "designation"]:
        for nom_anomalie, motif in anomaliesàsupprimer.items():
            if dataframe[colonne].str.contains(motif).any():
                print(f"Anomalie détectée dans la colonne '{colonne}' du dataframe {nom_dataframe}: {nom_anomalie}")
                print("Le nombre d'anomalies détectées est :", dataframe[colonne].str.contains(motif).sum())
            else:
                print(f"Aucune anomalie détectée dans la colonne '{colonne}' du dataframe {nom_dataframe}: {nom_anomalie}")


Aucune anomalie détectée dans la colonne 'description' du dataframe X_train: balises_html
Anomalie détectée dans la colonne 'description' du dataframe X_train: entites_html
Le nombre d'anomalies détectées est : 3
Anomalie détectée dans la colonne 'description' du dataframe X_train: caracteres_speciaux
Le nombre d'anomalies détectées est : 16624
Aucune anomalie détectée dans la colonne 'description' du dataframe X_train: espaces_inutiles
Aucune anomalie détectée dans la colonne 'designation' du dataframe X_train: balises_html
Aucune anomalie détectée dans la colonne 'designation' du dataframe X_train: entites_html
Anomalie détectée dans la colonne 'designation' du dataframe X_train: caracteres_speciaux
Le nombre d'anomalies détectées est : 7438
Aucune anomalie détectée dans la colonne 'designation' du dataframe X_train: espaces_inutiles
Aucune anomalie détectée dans la colonne 'description' du dataframe X_test: balises_html
Anomalie détectée dans la colonne 'description' du dataframe X_

In [5]:
#On déctecte encore un certaines nombre de caractères spéciaux mal encodés dans les colonnes description et designation des dataframes X_train_Rakuten et X_test_Rakuten, 
#nous allons donc les afficher pour étudier de quels sortes sont ces caractères.

for nom_dataframe, dataframe in [("X_train", X_train_Rakuten), ("X_test", X_test_Rakuten)]:
    for colonne in ["description", "designation"]:
        caracteresspeciaux = dataframe[colonne].str.findall(r"[^a-zA-Z0-9\sàâäéèêëîïôöùûüçÀÂÄÉÈÊËÎÏÔÖÙÛÜÇ.,'\-:;!?()/%€&+°\"]").explode().dropna().unique() #Puisque les anomalies détectées concernent des caractères spéciaux déclarés dans notre précédente bibliothèque d'anomalies à supprimer, nous allons les afficher pour les étudier et les corriger si nécessaire.
        if len(caracteresspeciaux) > 0:
            print(f"Caractères spéciaux restants dans la colonne '{colonne}' du dataframe {nom_dataframe}: {caracteresspeciaux}")
        else:
            print(f"Aucun caractère spécial restant dans la colonne '{colonne}' du dataframe {nom_dataframe}.")


Caractères spéciaux restants dans la colonne 'description' du dataframe X_train: ['²' '·' '¢' '¿' '*' '«' '»' 'ÿ' 'Ø' '=' '>' '_' '|' '×' 'ñ' '³' '–' '#'
 'ø' '~' '\\' '[' ']' 'œ' '<' '®' '@' 'º' '¡' '¯' '…' '`' '{' '}' 'ß' 'µ'
 'ó' '±' 'ª' '§' '•' '½' '£' '©' 'æ' '—' 'ã' 'á' 'í' '\u200b' 'Å' '´' 'Á'
 'Ú' 'ò' 'ú' '$' '¬' '\xad' '™' 'Ý' '⿿' 'ì' 'ă' 'ţ' 'å' '¥' 'Ñ' '¸' 'Ã'
 'а' '・' '¹' 'Ó' '♥' 'ſ' 'ο' '☆' '\u2fef' 'Œ' '^' 'Æ' '‡' '¼' '¾' '❤']
Caractères spéciaux restants dans la colonne 'designation' du dataframe X_train: ['_' '@' 'Ø' '®' '²' '|' '¿' '[' ']' 'ñ' '×' '*' '#' 'µ' 'º' '³' 'ó' '«'
 '»' '–' '$' '=' 'ø' 'ª' 'í' 'á' '\\' 'ã' '¢' 'ß' 'ÿ' '`' 'œ' 'Œ' 'å' 'Å'
 '©' '±' 'Ú' 'Ã' '★' 'ᵉ' 'ì' '>' 'Á' '♂' '\u200b' '§' '\xad' '¡' 'õ' '´'
 '\u200e' 'ò' '☆' '¥' '^' 'Æ' 'ō' 'ú' '—' '<']
Caractères spéciaux restants dans la colonne 'description' du dataframe X_test: ['×' '*' '=' '[' ']' '®' 'ó' '³' '±' '«' '»' 'Ø' 'µ' '¿' '_' 'œ' '~' '>'
 '|' '²' '#' '·' '¢' '…' '`' 'í' 'á' '¡' 'ñ' 'º' '\\' 

In [6]:
# Suite à l'affichage des caractères spéciaux restants dans les colonnes description et designation des dataframes X_train_Rakuten et X_test_Rakuten, 
# nous constatons que certains caractères invisibles ou parasites sont encore présents, par exemple :
#       \u200b : espace de largeur nulle (zero-width space), caractère invisible qui peut coller deux mots
#       \u200e : marque de direction gauche à droite (left-to-right mark), caractère de contrôle invisible
#       \xad   : trait d'union conditionnel (soft hyphen), caractère invisible qui peut couper un mot en deux
#       \u2fef : code Unicode non attribué, caractère sans signification
#       \uf0a7 : puce de liste issue de Word (police Wingdings), située dans la zone Unicode à usage privé

# Ces caractères n'apportent aucune information et peuvent déformer les mots lors de la tokenisation :
# ils sont donc supprimés.

# Nous allons également:
#       Remplacer le caractère spécial "×" par "x"
#       Remplacer le caractère spécial "´" par "'"
#       Remplacer le caractère spécial "а" par "a"
#       Remplacer le caractère spécial "ο" par "o"
#       Remplacer le caractère spécial "œ" par "oe"
#       Remplacer le caractère spécial "Œ" par "OE"
#       Remplacer le caractère spécial "æ" par "ae"
#       Remplacer le caractère spécial "Æ" par "AE"

for nom_dataframe, dataframe in [("X_train", X_train_Rakuten), ("X_test", X_test_Rakuten)]:
    for colonne in ["description", "designation"]:
        dataframe[colonne] = dataframe[colonne].str.replace(r"[\u200b\u200e\uf0a7\u2fef\u2fff\xad]", "", regex=True)  # Supprimer les caractères spéciaux restants correspondant aux codes Unicode \u200b, \u200e et \xad
        dataframe[colonne]=dataframe[colonne].str.replace("×","x",regex=False) # Remplacer le caractère spécial "×" par "x"
        dataframe[colonne]=dataframe[colonne].str.replace("´","'",regex=False) # Remplacer le caractère spécial "´" par "'"
        dataframe[colonne]=dataframe[colonne].str.replace("\u0430","a",regex=False) # Remplacer le caractère spécial "а" par "a"
        dataframe[colonne]=dataframe[colonne].str.replace("\u03bf","o",regex=False) # Remplacer le caractère spécial "ο" par "o"
        dataframe[colonne]=dataframe[colonne].str.replace("œ","oe",regex=False) # Remplacer le caractère spécial "œ" par "oe"
        dataframe[colonne]=dataframe[colonne].str.replace("Œ","OE",regex=False) # Remplacer le caractère spécial "Œ" par "OE"
        dataframe[colonne]=dataframe[colonne].str.replace("æ","ae",regex=False) # Remplacer le caractère spécial "æ" par "ae"
        dataframe[colonne]=dataframe[colonne].str.replace("Æ","AE",regex=False) # Remplacer le caractère spécial "Æ" par "AE"
        dataframe[colonne] = dataframe[colonne].str.replace("ᵉ", "e", regex=False)                     # e en exposant → e

# Revérification de la propreté des colonnes description et designation après la mise à jour

for nom_dataframe, dataframe in [("X_train", X_train_Rakuten), ("X_test", X_test_Rakuten)]:
    for colonne in ["description", "designation"]:
        caracteresspeciaux = dataframe[colonne].str.findall(r"[^a-zA-Z0-9\sàâäéèêëîïôöùûüçÀÂÄÉÈÊËÎÏÔÖÙÛÜÇ.,'\-:;!?()/%€&+°\"]").explode().dropna().unique()
        if len(caracteresspeciaux) > 0:
            print(f"Caractères spéciaux restants dans la colonne '{colonne}' du dataframe {nom_dataframe}: {caracteresspeciaux}")
        else:
            print(f"Aucun caractère spécial restant dans la colonne '{colonne}' du dataframe {nom_dataframe}.")

# Onconsidère pour l'instant que ce nettoyage est suffisant nous pouvons passer à l'étape suivante

Caractères spéciaux restants dans la colonne 'description' du dataframe X_train: ['²' '·' '¢' '¿' '*' '«' '»' 'ÿ' 'Ø' '=' '>' '_' '|' 'ñ' '³' '–' '#' 'ø'
 '~' '\\' '[' ']' '<' '®' '@' 'º' '¡' '¯' '…' '`' '{' '}' 'ß' 'µ' 'ó' '±'
 'ª' '§' '•' '½' '£' '©' '—' 'ã' 'á' 'í' 'Å' 'Á' 'Ú' 'ò' 'ú' '$' '¬' '™'
 'Ý' 'ì' 'ă' 'ţ' 'å' '¥' 'Ñ' '¸' 'Ã' '・' '¹' 'Ó' '♥' 'ſ' '☆' '^' '‡' '¼'
 '¾' '❤']
Caractères spéciaux restants dans la colonne 'designation' du dataframe X_train: ['_' '@' 'Ø' '®' '²' '|' '¿' '[' ']' 'ñ' '*' '#' 'µ' 'º' '³' 'ó' '«' '»'
 '–' '$' '=' 'ø' 'ª' 'í' 'á' '\\' 'ã' '¢' 'ß' 'ÿ' '`' 'å' 'Å' '©' '±' 'Ú'
 'Ã' '★' 'ì' '>' 'Á' '♂' '§' '¡' 'õ' 'ò' '☆' '¥' '^' 'ō' 'ú' '—' '<']
Caractères spéciaux restants dans la colonne 'description' du dataframe X_test: ['*' '=' '[' ']' '®' 'ó' '³' '±' '«' '»' 'Ø' 'µ' '¿' '_' '~' '>' '|' '²'
 '#' '·' '¢' '…' '`' 'í' 'á' '¡' 'ñ' 'º' '\\' 'ß' '•' '@' '©' 'ú' '{' '}'
 '–' 'ª' '<' 'ø' '¯' 'ã' '^' '—' 'Å' 'ò' '§' 'Ã' '£' 'ÿ']
Caractères spéciaux restants dans

In [7]:
# Deuxième étape du Preprocessing: Création d'une colonne Presence_description dans les dataframes X_train_Rakuten et X_test_Rakuten pour indiquer si 
# la colonne description est renseignée ou non après le nettoyage

X_train_Rakuten["Presence_description"] = X_train_Rakuten["description"].notnull().astype(int)
X_test_Rakuten["Presence_description"] = X_test_Rakuten["description"].notnull().astype(int)

print(X_train_Rakuten["Presence_description"].value_counts())
print(X_test_Rakuten["Presence_description"].value_counts())

print("Le pourcentage de description manquante dans X_train_Rakuten est:", (X_train_Rakuten["Presence_description"]==0).mean()*100)
print("Le pourcentage de description manquante dans X_test_Rakuten est:", (X_test_Rakuten["Presence_description"]==0).mean()*100)

print(X_train_Rakuten.head(30))
print(X_test_Rakuten.head(30))


Presence_description
1    55116
0    29800
Name: count, dtype: int64
Presence_description
1    8926
0    4886
Name: count, dtype: int64
Le pourcentage de description manquante dans X_train_Rakuten est: 35.09350416882566
Le pourcentage de description manquante dans X_test_Rakuten est: 35.37503620040545
                                          designation                                        description   productid     imageid  Presence_description
0   Olivia: Personalisiertes Notizbuch / 150 Seite...                                                NaN  3804725264  1263597046                     0
1   Journal Des Arts (Le) N° 133 Du 28/09/2001 - L...                                                NaN   436067568  1008141237                     0
2   Grand Stylet Ergonomique Bleu Gamepad Nintendo...  PILOT STYLE Touch Pen de marque Speedlink est ...   201115110   938777978                     1
3   Peluche Donald - Europe - Disneyland 2000 (Mar...                                        

In [8]:
# Troisième étape du Préprocessing: Creation d'une colonne longueur_description et longueur_designation dans les dataframes X_train_Rakuten et X_test_Rakuten pour indiquer la longueur des textes dans les colonnes description et designation après le nettoyage

for dataframe in [X_train_Rakuten, X_test_Rakuten]:
    dataframe["longueur_designation"] = dataframe["designation"].apply(lambda x: len (str(x)) if pd.notnull(x) else 0)
    dataframe["longueur_description"] = dataframe["description"].apply(lambda x: len(str(x)) if pd.notnull(x) else 0)
    

print(X_train_Rakuten.head(30))
print(X_train_Rakuten.head(30))

                                          designation                                        description   productid     imageid  Presence_description  longueur_designation  longueur_description
0   Olivia: Personalisiertes Notizbuch / 150 Seite...                                                NaN  3804725264  1263597046                     0                    88                     0
1   Journal Des Arts (Le) N° 133 Du 28/09/2001 - L...                                                NaN   436067568  1008141237                     0                   206                     0
2   Grand Stylet Ergonomique Bleu Gamepad Nintendo...  PILOT STYLE Touch Pen de marque Speedlink est ...   201115110   938777978                     1                    76                   674
3   Peluche Donald - Europe - Disneyland 2000 (Mar...                                                NaN    50418756   457047496                     0                    63                     0
4                        

In [9]:
# Quatrième étape du preprocessing: Nous allons détecter les différentes langues utilisés pour chaque ligne des dataframes X_train et X_test 
# et créer une colonne avec la langue utilisée pour l'écriture de designation et descritpion

detecteurdelangue = LanguageDetectorBuilder.from_languages(
    Language.FRENCH, Language.ENGLISH, Language.GERMAN,
    Language.ITALIAN, Language.SPANISH, Language.PORTUGUESE, Language.DUTCH, Language.POLISH,
    Language.TURKISH, Language.VIETNAMESE, Language.INDONESIAN, 
).with_minimum_relative_distance(0.25).build() # Utilisant d'un outil de détection de langue en renseignant les langues préalablement identifiées lors de l'étape de datavisualisation des données (cf Data_Viz_PROJET_LIORA_notebook)

def detection_langue(text): # Réalisation d'une fonction permettant de dicerner 4 catégories: la bonne détection de la langue d'un texte (ou du moins la langue la plus employé dans le texte), l'absence de texte, les caractères non textuels (ponctuation etc..), une erreur dans la détection du texte
    if pd.isna(text):
        return "Absence de texte" # espace vide (Nombreux NaN dans la colonne description)

    if re.search(r"[^\W\d_]", text) is None: # Recherche au moins une lettre dans chaque case, si il n'en trouve pas on déduit qu'elle ne contient pas de texte et donc n'a pas de langue associée   
            return "Non textuel"
    try:
        return detecteurdelangue.detect_language_of(text).name # si les conditions d'avant n'ont pas été vérifiées, il cherche la langue corrrespondant au texte
    except Exception as messageerreur: 
        #print(f"Erreur lors de la détection de la langue pour le texte: {text}. Erreur: {messageerreur}")
        return "unknown"  # Si il est impossible de statuer sur une langue de manière précise, alors la fonction classe comme inconnue la langue utilisée (dans le cas où trop de langues sont présentent dans une description ou une désignation par exemple)

for dataframe in [X_train_Rakuten, X_test_Rakuten]:
    for colonne in ["designation","description"]:
        dataframe["langue_" + colonne] = dataframe[colonne].apply(detection_langue).str.lower() # On passe les noms de langues en minuscule pour pouvoir les utiliser comme paramètres lors de la tokenisation en prenant en compte la langue du texte détéctée

print (X_train_Rakuten.head(30))
print(X_train_Rakuten["Presence_description"].head(30))


                                          designation                                        description   productid     imageid  Presence_description  longueur_designation  longueur_description langue_designation langue_description
0   Olivia: Personalisiertes Notizbuch / 150 Seite...                                                NaN  3804725264  1263597046                     0                    88                     0             german   absence de texte
1   Journal Des Arts (Le) N° 133 Du 28/09/2001 - L...                                                NaN   436067568  1008141237                     0                   206                     0             french   absence de texte
2   Grand Stylet Ergonomique Bleu Gamepad Nintendo...  PILOT STYLE Touch Pen de marque Speedlink est ...   201115110   938777978                     1                    76                   674             french             french
3   Peluche Donald - Europe - Disneyland 2000 (Mar...               

In [10]:
print(X_train_Rakuten.columns.tolist())

['designation', 'description', 'productid', 'imageid', 'Presence_description', 'longueur_designation', 'longueur_description', 'langue_designation', 'langue_description']


In [11]:
# Mise en minuscule des colonnes description et designation des dataframes X_train_Rakuten et X_test_Rakuten avant la tokenisation

for dataframe in [X_train_Rakuten, X_test_Rakuten]:
    for colonne in ["description", "designation"]:
        dataframe[colonne + "_lower"] = dataframe[colonne].str.lower()

In [12]:
# Remplacement des tirets par des espaces dans les colonnes description et designation des dataframes X_train_Rakuten et X_test_Rakuten

for dataframe in [X_train_Rakuten, X_test_Rakuten]:
    for colonne in ["description", "designation"]:
        dataframe[colonne + "_lower"] = dataframe[colonne].str.lower().str.replace("-", " ", regex=False)
        

In [13]:
# Tokenisation des colonnes description et designation des dataframes X_train_Rakuten et X_test_Rakuten

def tokeniser_langue(texte,langue): # Ecriture d'une fonction de tokenisation en fonction de la langue détécter sur chaque colonne designation ou description
    if not isinstance (texte, str):
        return([])
    if langue not in ["french", "english", "german", "italian", "spanish", "portuguese", "dutch", "polish", "turkish"]:
        langue = "french"  
    tokens = word_tokenize(texte, language=langue)
    return (tokens)


for dataframe in [X_train_Rakuten, X_test_Rakuten]:
    for colonne in ["description_lower", "designation_lower"]:
        dataframe[colonne + "_tokens"] = dataframe.apply(lambda row: tokeniser_langue(row[colonne], row["langue_" + colonne.split("_")[0]]), axis=1) # On tokenise ici en utilisant la fonction créé ci-dessus, le principe est ici d'utiliser le noms des colonnes description lower et designation lower en coupan le nom de ces dernier grâce à la fonction split pour obtenir in fine langue_description ou designation
        dataframe[colonne + "_tokens"] = dataframe[colonne + "_tokens"].apply(lambda tokens: [t for t in tokens if t.isalnum()]) # On ne garde ici que les tokens qui sont des chiffres ou des lettres 

print(X_train_Rakuten.iloc[:30, -5:])

# La ligne précédente nous permet d'afficher les colonnes résultantes de la tokenisation, on observe que certains tokens ne contiennent qu'une lettre (comme 'x' par exemple sur la ligne 7 de X_train_Rakuten), nous décidons de rajouter une ligne de code pour supprimer les tokens qui ne correspondent qu'à une seule lettre, MAIS nous garderons les tokens avec un seul chiffre par exemple, pour cela utilison la fonction isdigit() 

for dataframe in [X_train_Rakuten, X_test_Rakuten]:
    for colonne in ["description_lower", "designation_lower"]:
        dataframe[colonne + "_tokens"] = dataframe[colonne + "_tokens"].apply(lambda tokens:[t for t in tokens if len(t)>1 or t.isdigit()] )

print(X_train_Rakuten.iloc[:30, -5:]) # Vérifions que les lettres isolées ont disparues

   langue_description                                  description_lower                                  designation_lower                           description_lower_tokens                           designation_lower_tokens
0    absence de texte                                                NaN  olivia: personalisiertes notizbuch / 150 seite...                                                 []  [olivia, personalisiertes, notizbuch, 150, sei...
1    absence de texte                                                NaN  journal des arts (le) n° 133 du 28/09/2001   l...                                                 []  [journal, des, arts, le, 133, du, et, son, mar...
2              french  pilot style touch pen de marque speedlink est ...  grand stylet ergonomique bleu gamepad nintendo...  [pilot, style, touch, pen, de, marque, speedli...  [grand, stylet, ergonomique, bleu, gamepad, ni...
3    absence de texte                                                NaN  peluche donald   europ

In [14]:
# Supprimer les stop words dans les colonnes description et designation des dataframes X_train_Rakuten et X_test_Rakuten après la tokenisation

stopword = (set(nltk.corpus.stopwords.words("french"))
              | set(nltk.corpus.stopwords.words("english")) | set(nltk.corpus.stopwords.words("german")) | set(nltk.corpus.stopwords.words("italian")) | set(nltk.corpus.stopwords.words("spanish")) | set(nltk.corpus.stopwords.words("portuguese")) )

for dataframe in [X_train_Rakuten, X_test_Rakuten]:
    for colonne in ["description_lower_tokens", "designation_lower_tokens"]:
        dataframe[colonne + "_no_stopwords"] = dataframe[colonne].apply(lambda tokens: [t for t in tokens if t not in stopword])

print(X_train_Rakuten["description_lower_tokens_no_stopwords"].head(30))

0                                                    []
1                                                    []
2     [pilot, style, touch, pen, marque, speedlink, ...
3                                                    []
4     [luc, idées, grandeur, veut, organiser, jeu, g...
5                                                    []
6                                                    []
7     [conquerant, classique, cahier, 240, 320, mm, ...
8                                                    []
9     [tente, pliante, v3s5, pro, pvc, 500, 3, parti...
10    [timeless, dsw, seat, paired, types, legs, woo...
11    [canapé, fauteuil, chesterfield, cuir, antique...
12    [pcan, 2, year, old, play, game, yes, game, 2,...
13    [paire, voilages, droits, fantaisie, mettra, i...
14    [matelas, accueil, ferme, soutien, très, ferme...
15    [pince, agrafeuse, 591, capot, abs, dispositif...
16                                                   []
17                                              

In [15]:
#Lemmatisation des colonnes description et designation des dataframes X_train_Rakuten et X_test_Rakuten en fonction de la langue détectée dans la colonne langue_description et langue_description

def lemmatize_tokens(tokens, lang):
    if lang == "french":
        return [simplemma.lemmatize(t, lang=("fr")).lower() for t in tokens]
    elif lang == "english":
        return [simplemma.lemmatize(t, lang=("en","fr")).lower() for t in tokens]
    elif lang == "german":
        return [simplemma.lemmatize(t, lang=("de","fr")).lower() for t in tokens]
    elif lang == "italian":
        return [simplemma.lemmatize(t, lang=("it","fr")).lower() for t in tokens]
    elif lang == "spanish":
        return [simplemma.lemmatize(t, lang=("es","fr")).lower() for t in tokens]
    elif lang == "portuguese":
        return [simplemma.lemmatize(t, lang=("pt","fr")).lower() for t in tokens]
    else:
        return [simplemma.lemmatize(t, lang=("fr")).lower() for t in tokens] 

for dataframe in [X_train_Rakuten, X_test_Rakuten]:
    for colonne in ["description_lower_tokens_no_stopwords", "designation_lower_tokens_no_stopwords"]:
        dataframe[colonne + "_lemmatized"] = dataframe.apply(
            lambda row: lemmatize_tokens(row[colonne], row["langue_" + colonne.split("_")[0]]), axis=1
        )

print (X_train_Rakuten["description_lower_tokens_no_stopwords_lemmatized"].head(50))

0                                                    []
1                                                    []
2     [pilot, style, touch, pen, marque, speedlink, ...
3                                                    []
4     [luc, idée, grandeur, vouloir, organiser, jeu,...
5                                                    []
6                                                    []
7     [conquerant, classique, cahier, 240, 320, mm, ...
8                                                    []
9     [tente, pliant, v3s5, pro, pvc, 500, 3, partic...
10    [timeless, dsw, seat, pair, type, leg, wood, l...
11    [canapé, fauteuil, chesterfield, cuir, antique...
12    [pcan, 2, year, old, play, game, yes, game, 2,...
13    [paire, voilage, droit, fantaisie, mettre, int...
14    [matelas, accueil, ferme, soutien, très, ferme...
15    [pince, agrafeuse, 591, capot, abs, dispositif...
16                                                   []
17                                              

In [16]:
# Enlever les accents pour éviter la redondance des tokens dans les colonnes description et designation des dataframes X_train_Rakuten et X_test_Rakuten

for dataframe in [X_train_Rakuten, X_test_Rakuten]:
    for colonne in ["description_lower_tokens_no_stopwords_lemmatized", "designation_lower_tokens_no_stopwords_lemmatized", "description_lower_tokens", "designation_lower_tokens"]:
        dataframe[colonne] = dataframe[colonne].apply(lambda tokens: [unidecode(t) for t in tokens]) # Utilisation de la fonction unidecode pour enelever les accents

print (X_train_Rakuten["description_lower_tokens_no_stopwords_lemmatized"].head(50))

0                                                    []
1                                                    []
2     [pilot, style, touch, pen, marque, speedlink, ...
3                                                    []
4     [luc, idee, grandeur, vouloir, organiser, jeu,...
5                                                    []
6                                                    []
7     [conquerant, classique, cahier, 240, 320, mm, ...
8                                                    []
9     [tente, pliant, v3s5, pro, pvc, 500, 3, partic...
10    [timeless, dsw, seat, pair, type, leg, wood, l...
11    [canape, fauteuil, chesterfield, cuir, antique...
12    [pcan, 2, year, old, play, game, yes, game, 2,...
13    [paire, voilage, droit, fantaisie, mettre, int...
14    [matelas, accueil, ferme, soutien, tres, ferme...
15    [pince, agrafeuse, 591, capot, abs, dispositif...
16                                                   []
17                                              

In [17]:
# Créer une colonne combinant les tokens de description et designation pour les dataframes X_train_Rakuten et X_test_Rakuten

for dataframe in [X_train_Rakuten, X_test_Rakuten]:
    dataframe["tokens_combined_designation_description_sans_stopwords"] =  dataframe["designation_lower_tokens_no_stopwords_lemmatized"] + dataframe["description_lower_tokens_no_stopwords_lemmatized"]
    dataframe["tokens_combined_designation_description_brut"] = dataframe["designation_lower_tokens"] + dataframe["description_lower_tokens"]
      

In [18]:
print(X_train_Rakuten.shape[0], Y_train_CVw08PX.shape[0])          # On vérifie que aucune ligne n'a été perdu pendant le preprocessing
if X_train_Rakuten.index.equals(Y_train_CVw08PX.index):
    print("Index similaire entre les deux dataframes") 

84916 84916
Index similaire entre les deux dataframes


In [19]:
# Analyse des doublons potentiels une fois la fusion faites entre la colonne designation et descritpion de X_train_Rakuten

df_doublons_verif=X_train_Rakuten.merge(Y_train_CVw08PX,left_index=True, right_index=True)
df_doublons_verif["tokens_combined_designation_description_sans_stopwords"]=df_doublons_verif["tokens_combined_designation_description_sans_stopwords"].apply(' '.join) # On transforme les tokens en texte pour pouvoir analyser les doublons
print(df_doublons_verif.duplicated(subset=['tokens_combined_designation_description_sans_stopwords','prdtypecode']).sum()) # On regarde si il y a des doublons pour le texte fusionné ET la classe respective de chaque produit


1703


In [20]:
# Suppression des doublons dans le datarame X_train_Rakuten en ne gardant que les indexs du dataframe intermédaire après nettoyage des doublons

df_sansdoublons_verif=df_doublons_verif.drop_duplicates(subset=['tokens_combined_designation_description_sans_stopwords', 'prdtypecode'], keep='first')
indexàgarder=df_sansdoublons_verif.index
X_train_Rakuten=X_train_Rakuten.loc[indexàgarder]
Y_train_CVw08PX = Y_train_CVw08PX.loc[indexàgarder]

print(len(X_train_Rakuten))
if len(X_train_Rakuten) + 1703==84916:
    print("Tous les doublons ont bien été supprimés")
else:
    print("Trop de lignes supprimées")

83213
Tous les doublons ont bien été supprimés


In [21]:
# La dernière partie du préprocessing pour la partie textuel de ce projet concerne la vectorisation TF-IDF. 
# En effet notre futur modèle ne comprend pas les données textueelles, il faut donc transformer ce texte en nombre.
# Pour ce faire nous allons utiliser le principe de la vectorisation TF-IDF. L'idée générale est de définir le 'poid' de chaque mot présent dans notre dataframes. 

# Ici nous avons préalablement fusionner les deux colonnes designation et description après la tokenisation, la suppression des stops words et la lemmatisation, pourquoi ? 
# Puisque la colonne description à beaucoup de données manquantes et que la colonne designation ets complète, la fusion des deux colonnes permet de fournir du texte pour tous les produits, 
# facilitant le travail de classification du modèle. 
# Cette fusion permet d'éviter la création de colonne multiple lors de la vectorisation correspondant au même mot qu'on retoruverais à la fois dans la description et la designation

# Après plusieurs recherches, la fonction de vectorisation ce présente comme ceci: 

# vectoriseur = TfidfVectorizer(analyzer=lambda x: x, min_df=2, max_df=0.9, sublinear_tf=True)

# min_df et max_df sont des seuils permettant de filtrer les mots trop rare ou trop abondants dans le dataset avant la vectorisation, cela permet de ne pas surcharger la matrice finale. 
# Afin de rechercher les meilleurs valeurs de ces deux hyper-paramètres dans notre contexte, nous allons utiliser la méthode GridSearchCV pour tester plusieurs combinaisons de paramètres et trouver la meilleure. 
# Dans un premier temps nous allons effectuer un train_split de nos données pour éviter une fuite de données éventuelles.

X_train, X_test, y_train, y_test = train_test_split( X_train_Rakuten["tokens_combined_designation_description_sans_stopwords"], Y_train_CVw08PX['prdtypecode'],test_size=0.2,stratify=Y_train_CVw08PX['prdtypecode'],random_state=42)

X_tr, X_val, y_tr, y_val = train_test_split(X_train, y_train,test_size=0.2,stratify=y_train,random_state=42)

min_df= [0.0, 0.00005, 0.0001, 0.0005, 0.001]
max_df= [0.5, 0.6, 0.7, 0.8, 0.9, 1.0]
liste_score_parametres=[] 

for elementmin in min_df:
    for elementmax in max_df:
        vectoriseur = TfidfVectorizer(analyzer=lambda x: x, min_df=elementmin, max_df=elementmax, sublinear_tf=True)
        X_tr_vec=vectoriseur.fit_transform(X_tr)
        X_tr_val=vectoriseur.transform(X_val)
        for param in [0.1, 0.5, 1, 2]: 
            modele = LinearSVC(C=param)
            modele.fit(X_tr_vec, y_tr)
            score=f1_score(y_val, modele.predict(X_tr_val),average="weighted")
            liste_score_parametres.append((elementmin,elementmax,score,param))

resultats_score=pd.DataFrame(liste_score_parametres,columns=["min_df", "max_df", "f1_score","paramètre C"])
resultats_score=resultats_score.sort_values("f1_score",ascending=False)
print(resultats_score)

# On sélectionne les meilleurs hyperparamètres min_df = 0 et max_df = 1 pour l'entrainement et le test final sur X_train,y_train et pour le test sur X_test






     min_df  max_df  f1_score  paramètre C
13    0.000     0.8  0.814333          0.5
5     0.000     0.6  0.814333          0.5
17    0.000     0.9  0.814333          0.5
21    0.000     1.0  0.814333          0.5
9     0.000     0.7  0.814333          0.5
..      ...     ...       ...          ...
112   0.001     0.9  0.768569          0.1
104   0.001     0.7  0.768569          0.1
116   0.001     1.0  0.768569          0.1
96    0.001     0.5  0.768569          0.1
100   0.001     0.6  0.768569          0.1

[120 rows x 4 columns]


In [22]:
# Maintenant que nous avons sélectionné les meilleurs paramètres pour min_df et max_df, nous allons passé à l'entrainement final sur X_train et y_train

vectoriseur = TfidfVectorizer(analyzer=lambda x: x, min_df=0.0, max_df=1.0, sublinear_tf=True)
X_tr_vec=vectoriseur.fit_transform(X_train)
X_tr_val=vectoriseur.transform(X_test)

modele = LinearSVC(C=0.5 )
modele.fit(X_tr_vec, y_train)

# Evaluation en utilisant X_test vectoriser (X_tr_val) et y_test

y_pred=modele.predict(X_tr_val)

score_final=f1_score(y_test,y_pred,average="weighted")
print(score_final)

# On obtient un f1_score de 0.82 cohérent avec le score obtenu précédemment lors de la séléction des paramètres min_df et max_df, 
# ce qui veut dire qu'avec ces paramètres de vectorisation, le modèle est capable de classifier correctement en grande majorité des produits inconnus 
# à partir de notre set de données vectorisé avec les paramètres min_df et max_df choisis.   



0.8201173547793713


In [23]:
# Observons maintenant un aperçus de notre matrice obtenu après vectorisation de notre set de train obtenu après le split de X_train_Rakuten et Y_train_CVw08PX 

motutilise= vectoriseur.get_feature_names_out()

apercu = pd.DataFrame(X_tr_vec[:10].toarray(), columns=motutilise)

print(apercu)

# Au vue des résultats, il semblerait qu'il y ai beaucoup de longues séries de chiffres, correspondant peut être à des références précises, 
# et qui n'apporterait pas grand chose à la prédiction de la classe produits, nous pourrions peut être utiliser une ligne de code supplémentaires 
# après la tokenisation pour ne garder que les nombres cohérent type années, dimensions, etc.. souvent inférieur ou égal à 5, 
# donc t for t in tokens if not (t.isdigit() and len(t) > 5)]


     0   00  000  0000  00000  000000  0000000  0000057743  00001  00005  0001  000195  0001ah  0002  000200br  00025  0003  00030  000353  00036  0004  0005  00050  00057  0006  000652  00069  0007  00075  0008  00085  00095  000967148201  000h  001  0011  0012  00120  00122  0013  0014  00140  \
0  0.0  0.0  0.0   0.0    0.0     0.0      0.0         0.0    0.0    0.0   0.0     0.0     0.0   0.0       0.0    0.0   0.0    0.0     0.0    0.0   0.0   0.0    0.0    0.0   0.0     0.0    0.0   0.0    0.0   0.0    0.0    0.0           0.0   0.0  0.0   0.0   0.0    0.0    0.0   0.0   0.0    0.0   
1  0.0  0.0  0.0   0.0    0.0     0.0      0.0         0.0    0.0    0.0   0.0     0.0     0.0   0.0       0.0    0.0   0.0    0.0     0.0    0.0   0.0   0.0    0.0    0.0   0.0     0.0    0.0   0.0    0.0   0.0    0.0    0.0           0.0   0.0  0.0   0.0   0.0    0.0    0.0   0.0   0.0    0.0   
2  0.0  0.0  0.0   0.0    0.0     0.0      0.0         0.0    0.0    0.0   0.0     0.0     0.0   0.0   